In [1]:
# Core libraries
import pandas as pd
import numpy as np
import json

#preprocessing
from sklearn import set_config
set_config(display = 'diagram', transform_output= "pandas")


# Visualization
import matplotlib.pyplot as plt
import seaborn as sns


# Project config
from src.params import *
from src.utils import *
from src.models.model_pipeline import *
from src.preprocess.preproc_pipeline import *

# Plot configuration
#%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
import warnings
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2

# I. ingest

In [2]:
airqual = load_data_local("../data/raw/airqual.csv", "airqual")
weather = load_data_local("../data/raw/weather.csv", "weather")

✅ Loaded 17951 rows from ../data/raw/airqual.csv
✅ Loaded 4386 rows from ../data/raw/weather.csv


# II. preprocess

In [3]:
dataset_metadata, X, y = preprocessing_pipeline(airqual_df= airqual, weather_df= weather)

⚙️  Starting preprocessing — airqual: 17951 rows, weather: 4386 rows
⚠️  162 aberrant (negative or zero) values found (0.90%)
✅ All negative values removed — 17789 rows remaining
  [gap filter] 4 sensor(s) flagged — max_gap=30d, max_q75=10.0d
  [coverage filter] 1 sensor(s) flagged — min_cov=70%, max_bad_months=20%
✅ filter_sensors: 4 sensor(s) removed, 22 remaining
✅ Sensors averaged — 4278 rows (city × date)
✅ DataFrames merged successfully — 4278/4386 days have airqual data (97.5%)
   rows after merge: 4386
total nan before: 108
✅ Imputing successful.1-day gaps before: 108. After: 55
   features generated (custom): 20 features
✅ dropna: 367 rows removed, 4019 remaining
   rows dropped (dropna + preprocess cols): 367 → 4019 remaining
✅ Preprocessing done — 4019 rows, 10 features | 2023-05-15 00:00:00 → 2025-04-29 00:00:00


# III. setup MLflow

In [4]:
setup_mlflow()

2026/03/06 19:43:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/03/06 19:43:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/03/06 19:43:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/03/06 19:43:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/03/06 19:43:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/03/06 19:43:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/03/06 19:43:27 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/06 19:43:27 INFO mlflow.store.db.utils: Updating database tables
2026/03/06 19:43:27 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/06 19:43:27 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/03/06 19:43:27 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2026/03/06 19:4

✅ MLflow ready
   tracking URI : sqlite:///mlflow.db
   experiment   : breathe-mlflow-experiment (id: 1)


# IV. First pass: train and eval first model

- should come out as v1 "champion"

In [5]:
trained_model, model_version = run_training(X,y, dataset_metadata)

ℹ️ MLflow run started — run_id: 043a37ad524b4688aeda2dbd9f172ed1
📦 Instantiating model...
⚙️  Fitting on 4019 rows, 10 features...


2026/03/06 19:43:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


   fit time: 0.5s
💾 Saving model to MLflow registry...


2026/03/06 19:43:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


   model registered as version 1
✅ Model v1 trained, logged and registered as 'champion'


Successfully registered model 'breathe-mlflow-model'.
Created version '1' of model 'breathe-mlflow-model'.


In [6]:
score = run_evaluating(X, y,
               dataset_metadata= dataset_metadata,
               model= trained_model,
               model_version= model_version,
               alias= "champion",
               eval_mode= "test_set")

score

ℹ️ MLflow run started — run_id: 488f0a77197f4e929bb69dd3465c527d
⚙️ Evaluating on 4019 rows (test_set)...
   RMSE: 0.2962
✅ Evaluation done — RMSE: 0.2962 | model v1 | test_set


0.2962083206563794

# V. 2nd pass: creation and eval second model
- this one should come out as v2 "challenger"

In [7]:
trained_model_v2, model_version_v2 = run_training(X,y, dataset_metadata)

ℹ️ MLflow run started — run_id: 9e87a0db1fb64ad49be1c3302d4ebc14
📦 Instantiating model...
⚙️  Fitting on 4019 rows, 10 features...


2026/03/06 19:44:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


   fit time: 0.48s
💾 Saving model to MLflow registry...


2026/03/06 19:44:10 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


   model registered as version 2
no champion found. registring model as first champion
✅ Model v2 trained, logged and registered as challenger


Registered model 'breathe-mlflow-model' already exists. Creating a new version of this model...
Created version '2' of model 'breathe-mlflow-model'.


In [8]:
score_2 = run_evaluating(X, y,
               dataset_metadata= dataset_metadata,
               model= trained_model_v2,
               model_version= model_version_v2,
               alias= "challenger",
               eval_mode= "test_set")

score_2

ℹ️ MLflow run started — run_id: 8958c58390864b89a6598c4f3a908615
⚙️ Evaluating on 4019 rows (test_set)...
   RMSE: 0.2962
✅ Evaluation done — RMSE: 0.2962 | model v2 | test_set


0.2962083206563794

# VI. testing promotion of challenger

In [9]:
client = MlflowClient()
promote_challenger(client)

   archived previous champion v1
✅ Challenger v2 promoted to champion
